# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We'll access the dataset through its Croissant schema, systematically review its structure (record sets, fields, and columns), and perform basic exploratory data analysis.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access to the dataset metadata
meta = dataset.metadata

print(f"Loaded dataset: {meta.name}")
print(f"Description: {meta.description}")

## 2. Data Overview
Review all available record sets, their fields, and their `@id` references for targeted extraction.

In [ ]:
# List all available record sets with their @id and display info
print("Available Record Sets and Fields:")
record_sets = meta.record_sets
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    # Fields may be a list of @ids or single dict
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field_ref in fields:
        field_id = field_ref['@id'] if isinstance(field_ref, dict) else field_ref
        field_details = meta.find_by_id(field_id)
        print(f"    - {field_id}: {field_details.get('name', '(no name)')}")

## 3. Data Extraction 
Extract data from one or more available record sets (using their `@id`), load into DataFrames, and inspect the structure. Use the actual record set and field `@id`s identified above.

In [ ]:
# Here we list record set @ids for which to extract data
record_set_ids = [rs['@id'] for rs in meta.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, grouping, and other data preparation steps. Reference all columns through their Croissant `@id`. Replace the placeholders below with actual field `@id`s you found above as appropriate for this dataset.

In [ ]:
# === Example for EDA on a selected numeric field and grouping field ===
# Please update these IDs based on the previous overview section

# Example (replace with correct @id):
# record_set_id = 'cr:RecordSet_1'
# numeric_field_id = 'cr:column:coverage_percent'
# group_field_id = 'cr:column:sample_type'

# For this demonstration, let's automatically select the first numeric field (if any) for the first record set
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
numeric_col = None
# Try to find a likely numeric column by name (if dataset names hint at numeric data)
for col in df.columns:
    if ('coverage' in col or 'MW' in col or 'count' in col or 'abundance' in col or 'peptide' in col or 'intensity' in col) and pd.api.types.is_numeric_dtype(df[col]):
        numeric_col = col
        break
if numeric_col is None:
    # Or just pick the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break

# Choose a candidate group field
group_field = None
for col in df.columns:
    if ('type' in col or 'group' in col or 'sample' in col) and df[col].nunique() < max(10, 0.2*len(df)):
        group_field = col
        break

if numeric_col is not None:
    # EDA example steps
    threshold = df[numeric_col].mean() if pd.api.types.is_numeric_dtype(df[numeric_col]) else 10
    filtered_df = df[df[numeric_col] > threshold]
    print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_col}_normalized"] = (
        filtered_df[numeric_col] - filtered_df[numeric_col].mean()
    ) / filtered_df[numeric_col].std()
    print(f"\nNormalized {numeric_col} for filtered records:")
    print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())
    
    # Group by field, if such a field exists
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_col].mean()
        print(f"\nMean {numeric_col} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Plot a distribution or relationship for numeric variables using Pandas/Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_col is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_col].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_col} in {record_set_id}")
    plt.xlabel(numeric_col)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_col, data=df)
        plt.title(f"{numeric_col} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
- This notebook demonstrated how to explore and analyze a Croissant-compliant dataset using the `mlcroissant` library.
- We loaded metadata, identified available record sets and their fields using `@id`, loaded the corresponding tabular records, and performed basic exploratory data analysis with visualizations.
- For further analyses, consult the schema documentation or field definitions, and tailor the filtering/grouping to your biological or analytical questions.

<!-- For more information see [mlcroissant documentation](https://mlcommons.github.io/croissant/python/reference/) -->